In [2]:
import torch
import numpy as np

from flash_ansr.eval.metrics import bootstrapped_metric_ci
import matplotlib.pyplot as plt

In [3]:
def x_prior(n, input_dim):
    # Sample bounds a and b from cauchy in each dimension
    a = torch.distributions.Cauchy(0, 1).sample((input_dim, ))
    b = torch.distributions.Cauchy(0, 1).sample((input_dim,))
    low = torch.min(a, b)
    high = torch.max(a, b)

    # Within a and b, sample uniformly
    return torch.distributions.Uniform(low, high).sample((n,))

def c_prior(n, input_dim):
    return torch.distributions.Cauchy(0, 1).sample((n, input_dim))

In [4]:
def measure_capability(
    function_sampler,
    input_dim,
    n_probes=100,
    n_models=100,
    n_repeats=100,
    n_bootstrap=100,
    max_resample=10,
    shared_probes=False,
    shared_probes_across_repeats=False,
    dtype=torch.float32,
    use_gram_if_smaller=True,
    ):
    """
    Faster effective-dimension estimator using lower precision and a Gram option.

    The API matches measure_capability with optional speed controls.
    """
    eff_dims = []
    X_probe_global = None
    if shared_probes and shared_probes_across_repeats:
        X_probe_global = x_prior(n_probes, input_dim).to(dtype)

    for _ in range(n_repeats):
        outputs = []
        X_probe_shared = None
        if shared_probes:
            if X_probe_global is not None:
                X_probe_shared = X_probe_global
            else:
                X_probe_shared = x_prior(n_probes, input_dim).to(dtype)
        for _ in range(n_models):
            y = None
            for _ in range(max_resample):
                func = function_sampler()
                with torch.no_grad():
                    if X_probe_shared is None:
                        X_probe = x_prior(n_probes, input_dim).to(dtype)
                    else:
                        X_probe = X_probe_shared
                    candidate = func(X_probe).flatten()
                if torch.isfinite(candidate).all():
                    y = candidate
                    break
            if y is not None:
                outputs.append(y)

        if len(outputs) < 2:
            eff_dims.append(0.0)
            continue

        Y = torch.stack(outputs)
        if torch.std(Y, dim=0).sum() < 1e-9:
            eff_dims.append(0.0)
            continue

        Y_centered = Y - Y.mean(dim=0, keepdim=True)
        n_models_eff, n_probes_eff = Y_centered.shape
        if use_gram_if_smaller and n_models_eff <= n_probes_eff:
            gram = (Y_centered @ Y_centered.T) / (n_models_eff - 1)
            eig_vals = torch.linalg.eigvalsh(gram)
        else:
            svals = torch.linalg.svdvals(Y_centered)
            eig_vals = (svals ** 2) / (n_models_eff - 1)
        eig_vals = eig_vals[eig_vals > 1e-9]
        if len(eig_vals) == 0:
            eff_dims.append(0.0)
            continue

        sum_vals = eig_vals.sum()
        sum_sq_vals = (eig_vals ** 2).sum()
        eff_dim = (sum_vals ** 2) / sum_sq_vals
        eff_dims.append(eff_dim.item())

    eff_dims = np.array(eff_dims)
    return bootstrapped_metric_ci(eff_dims, metric=np.nanmedian, n=n_bootstrap)

In [5]:
from tqdm import tqdm


def measure_capacity_var(
    function_sampler,
    input_dim,
    n_probes=100,
    n_models=100,
    n_repeats=100,
    n_bootstrap=100,
    n_probe_sets=None,
    max_resample=10,
    vary_probes=True,
    vary_models=True,
    shared_probes=False,
    dtype=torch.float32,
    ):
    """
    Measures the variance of the per-sample output variance across probes and/or models.

    If vary_probes is True, multiple datasets X are sampled (probe sets).
    If vary_models is True, multiple function instances are sampled.
    """
    if n_probe_sets is None:
        n_probe_sets = n_models

    n_models_eff = n_models if vary_models else 1
    n_probe_sets_eff = n_probe_sets if vary_probes else 1

    total_combinations = n_models_eff * n_probe_sets_eff * n_repeats
    pbar = tqdm(total=total_combinations, desc="Measuring capacity var")

    var_of_vars = []
    for _ in range(n_repeats):
        var_samples = []
        X_shared = None
        if not vary_probes or shared_probes:
            X_shared = x_prior(n_probes, input_dim).to(dtype)

        for _ in range(n_models_eff):
            func = function_sampler()
            y_probe_sets = None
            for _ in range(max_resample):
                if vary_probes and X_shared is None:
                    X_probe_sets = x_prior(n_probes * n_probe_sets_eff, input_dim).to(dtype)
                    X_probe_sets = X_probe_sets.reshape(n_probe_sets_eff, n_probes, input_dim)
                else:
                    X_probe_sets = X_shared.unsqueeze(0).expand(n_probe_sets_eff, -1, -1)

                X_flat = X_probe_sets.reshape(-1, input_dim)
                with torch.no_grad():
                    candidate = func(X_flat).reshape(-1)
                if candidate.numel() == n_probe_sets_eff * n_probes and torch.isfinite(candidate).all():
                    y_probe_sets = candidate.reshape(n_probe_sets_eff, n_probes)
                    break
                if vary_models:
                    func = function_sampler()
            if y_probe_sets is not None:
                var_samples.extend(torch.var(y_probe_sets, dim=1, unbiased=False).tolist())
            pbar.update(n_probe_sets_eff)

        if len(var_samples) < 2:
            var_of_vars.append(0.0)
            continue
        var_tensor = torch.tensor(var_samples, dtype=dtype)
        var_of_vars.append(torch.var(var_tensor, unbiased=False).item())

    var_of_vars = np.array(var_of_vars)
    return bootstrapped_metric_ci(var_of_vars, metric=np.nanmedian, n=n_bootstrap)

In [6]:
def linear_sampler():
    c = c_prior(1, 1).squeeze()
    return lambda x: c * x

median_fresh, lower_fresh, upper_fresh = measure_capability(
    linear_sampler,
    input_dim=1,
    shared_probes=False,
 )
median_shared, lower_shared, upper_shared = measure_capability(
    linear_sampler,
    input_dim=1,
    shared_probes=True,
 )
print(
    f'Effective Dimension (fresh probes): {median_fresh:.2f} '
    f'(95% CI: {lower_fresh:.2f} -- {upper_fresh:.2f})'
 )
print(
    f'Effective Dimension (shared probes): {median_shared:.2f} '
    f'(95% CI: {lower_shared:.2f} -- {upper_shared:.2f})'
 )

Effective Dimension (fresh probes): 1.22 (95% CI: 1.15 -- 1.27)
Effective Dimension (shared probes): 1.00 (95% CI: 1.00 -- 1.00)


In [7]:
def affine_sampler():
    c0, c1 = c_prior(1, 2).squeeze()
    return lambda x: c0 + c1 * x

median_fresh, lower_fresh, upper_fresh = measure_capability(
    affine_sampler,
    input_dim=1,
    shared_probes=False,
 )
median_shared, lower_shared, upper_shared = measure_capability(
    affine_sampler,
    input_dim=1,
    shared_probes=True,
 )
print(
    f'Effective Dimension (fresh probes): {median_fresh:.2f} '
    f'(95% CI: {lower_fresh:.2f} -- {upper_fresh:.2f})'
 )
print(
    f'Effective Dimension (shared probes): {median_shared:.2f} '
    f'(95% CI: {lower_shared:.2f} -- {upper_shared:.2f})'
 )

Effective Dimension (fresh probes): 1.20 (95% CI: 1.15 -- 1.23)
Effective Dimension (shared probes): 1.02 (95% CI: 1.01 -- 1.04)


In [8]:
def over_polynomial_sampler():
    c0, c1, c2 = c_prior(1, 3).squeeze()
    return lambda x: c0 + c1 * c2 * x

median_fresh, lower_fresh, upper_fresh = measure_capability(
    over_polynomial_sampler,
    input_dim=1,
    shared_probes=False,
 )
median_shared, lower_shared, upper_shared = measure_capability(
    over_polynomial_sampler,
    input_dim=1,
    shared_probes=True,
 )
print(
    f'Effective Dimension (fresh probes): {median_fresh:.2f} '
    f'(95% CI: {lower_fresh:.2f} -- {upper_fresh:.2f})'
 )
print(
    f'Effective Dimension (shared probes): {median_shared:.2f} '
    f'(95% CI: {lower_shared:.2f} -- {upper_shared:.2f})'
 )

Effective Dimension (fresh probes): 1.17 (95% CI: 1.13 -- 1.23)
Effective Dimension (shared probes): 1.01 (95% CI: 1.00 -- 1.02)


In [9]:
def linear_sampler_2d():
    c0, c1 = c_prior(1, 2).squeeze()
    return lambda x: c0 * x[:, 0] + c1 * x[:, 1]

median_fresh, lower_fresh, upper_fresh = measure_capability(
    linear_sampler_2d,
    input_dim=2,
    shared_probes=False,
 )
median_shared, lower_shared, upper_shared = measure_capability(
    linear_sampler_2d,
    input_dim=2,
    shared_probes=True,
 )
print(
    f'Effective Dimension (fresh probes): {median_fresh:.2f} '
    f'(95% CI: {lower_fresh:.2f} -- {upper_fresh:.2f})'
 )
print(
    f'Effective Dimension (shared probes): {median_shared:.2f} '
    f'(95% CI: {lower_shared:.2f} -- {upper_shared:.2f})'
 )

Effective Dimension (fresh probes): 1.23 (95% CI: 1.15 -- 1.32)
Effective Dimension (shared probes): 1.03 (95% CI: 1.02 -- 1.05)


In [10]:
def linear_quad_sampler_2d():
    c0, c1 = c_prior(1, 2).squeeze()
    return lambda x: c0 * x[:, 0] + c1 * x[:, 1]**2

median_fresh, lower_fresh, upper_fresh = measure_capability(
    linear_quad_sampler_2d,
    input_dim=2,
    shared_probes=False,
 )
median_shared, lower_shared, upper_shared = measure_capability(
    linear_quad_sampler_2d,
    input_dim=2,
    shared_probes=True,
 )
print(
    f'Effective Dimension (fresh probes): {median_fresh:.2f} '
    f'(95% CI: {lower_fresh:.2f} -- {upper_fresh:.2f})'
 )
print(
    f'Effective Dimension (shared probes): {median_shared:.2f} '
    f'(95% CI: {lower_shared:.2f} -- {upper_shared:.2f})'
 )

Effective Dimension (fresh probes): 1.10 (95% CI: 1.04 -- 1.18)
Effective Dimension (shared probes): 1.01 (95% CI: 1.01 -- 1.01)


In [11]:
def linear_sampler_3d():
    c0, c1, c2 = c_prior(1, 3).squeeze()
    return lambda x: c0 * x[:, 0] + c1 * x[:, 1] + c2 * x[:, 2]

median_fresh, lower_fresh, upper_fresh = measure_capability(
    linear_sampler_3d,
    input_dim=3,
    shared_probes=False,
 )
median_shared, lower_shared, upper_shared = measure_capability(
    linear_sampler_3d,
    input_dim=3,
    shared_probes=True,
 )
print(
    f'Effective Dimension (fresh probes): {median_fresh:.2f} '
    f'(95% CI: {lower_fresh:.2f} -- {upper_fresh:.2f})'
 )
print(
    f'Effective Dimension (shared probes): {median_shared:.2f} '
    f'(95% CI: {lower_shared:.2f} -- {upper_shared:.2f})'
 )

Effective Dimension (fresh probes): 1.23 (95% CI: 1.16 -- 1.29)
Effective Dimension (shared probes): 1.07 (95% CI: 1.04 -- 1.14)


In [12]:
def linear_affine_sampler_3d():
    c0, c1, c2, c3 = c_prior(1, 4).squeeze()
    return lambda x: c0 * x[:, 0] + c1 * x[:, 1] + c2 * x[:, 2] + c3

median_fresh, lower_fresh, upper_fresh = measure_capability(
    linear_affine_sampler_3d,
    input_dim=3,
    shared_probes=False,
 )
median_shared, lower_shared, upper_shared = measure_capability(
    linear_affine_sampler_3d,
    input_dim=3,
    shared_probes=True,
 )
print(
    f'Effective Dimension (fresh probes): {median_fresh:.2f} '
    f'(95% CI: {lower_fresh:.2f} -- {upper_fresh:.2f})'
 )
print(
    f'Effective Dimension (shared probes): {median_shared:.2f} '
    f'(95% CI: {lower_shared:.2f} -- {upper_shared:.2f})'
 )

Effective Dimension (fresh probes): 1.22 (95% CI: 1.18 -- 1.28)
Effective Dimension (shared probes): 1.07 (95% CI: 1.05 -- 1.14)


In [13]:
def sine_sampler():
    return lambda x: torch.sin(x)

median_fresh, lower_fresh, upper_fresh = measure_capability(
    sine_sampler,
    input_dim=1,
    shared_probes=False,
 )
median_shared, lower_shared, upper_shared = measure_capability(
    sine_sampler,
    input_dim=1,
    shared_probes=True,
 )
print(
    f'Effective Dimension (fresh probes): {median_fresh:.2f} '
    f'(95% CI: {lower_fresh:.2f} -- {upper_fresh:.2f})'
 )
print(
    f'Effective Dimension (shared probes): {median_shared:.2f} '
    f'(95% CI: {lower_shared:.2f} -- {upper_shared:.2f})'
 )

Effective Dimension (fresh probes): 3.71 (95% CI: 3.59 -- 3.80)
Effective Dimension (shared probes): 0.00 (95% CI: 0.00 -- 0.00)


In [14]:
def affine_sine_sampler():
    c0, c1, c2, c3 = c_prior(1, 4).squeeze()
    return lambda x: c0 * torch.sin(c1 * x[:, 0] + c2) + c3

median_fresh, lower_fresh, upper_fresh = measure_capability(
    affine_sine_sampler,
    input_dim=1,
    shared_probes=False,
 )
median_shared, lower_shared, upper_shared = measure_capability(
    affine_sine_sampler,
    input_dim=1,
    shared_probes=True,
 )
print(
    f'Effective Dimension (fresh probes): {median_fresh:.2f} '
    f'(95% CI: {lower_fresh:.2f} -- {upper_fresh:.2f})'
 )
print(
    f'Effective Dimension (shared probes): {median_shared:.2f} '
    f'(95% CI: {lower_shared:.2f} -- {upper_shared:.2f})'
 )

Effective Dimension (fresh probes): 1.15 (95% CI: 1.06 -- 1.25)
Effective Dimension (shared probes): 1.04 (95% CI: 1.02 -- 1.09)


In [15]:
# def cos_exp_sampler_2d():
#     c0, c1 = c_prior(1, 2).squeeze()
#     return lambda x: torch.exp(-torch.abs(c0) * x[:, 0]) * torch.cos(c1 * x[:, 1])

# median_fresh, lower_fresh, upper_fresh = measure_capability(
#     cos_exp_sampler_2d,
#     input_dim=2,
#     shared_probes=False,
#  )
# median_shared, lower_shared, upper_shared = measure_capability(
#     cos_exp_sampler_2d,
#     input_dim=2,
#     shared_probes=True,
#  )
# print(
#     f'Effective Dimension (fresh probes): {median_fresh:.2f} '
#     f'(95% CI: {lower_fresh:.2f} -- {upper_fresh:.2f})'
#  )
# print(
#     f'Effective Dimension (shared probes): {median_shared:.2f} '
#     f'(95% CI: {lower_shared:.2f} -- {upper_shared:.2f})'
#  )

In [6]:
capacity_var_cfg = dict(
    n_probes=100,
    n_models=100,
    n_repeats=100,
    n_bootstrap=100,
    max_resample=10,
    dtype=torch.float32,
 )

def _run_capacity_var_triplet(name, sampler, input_dim):
    print(f"\n{name} (input_dim={input_dim})")
    for vary_probes, vary_models in [(True, True), (True, False), (False, True)]:
        median, lower, upper = measure_capacity_var(
            sampler,
            input_dim=input_dim,
            vary_probes=vary_probes,
            vary_models=vary_models,
            **capacity_var_cfg,
        )
        print(
            f"Var(Var(y)) probes={vary_probes}, models={vary_models}: "
            f"{median:.4f} (95% CI: {lower:.4f} -- {upper:.4f})"
        )

_run_capacity_var_triplet("linear_sampler", linear_sampler, 1)
_run_capacity_var_triplet("affine_sampler", affine_sampler, 1)
_run_capacity_var_triplet("over_polynomial_sampler", over_polynomial_sampler, 1)
_run_capacity_var_triplet("linear_sampler_2d", linear_sampler_2d, 2)
_run_capacity_var_triplet("linear_quad_sampler_2d", linear_quad_sampler_2d, 2)
_run_capacity_var_triplet("linear_sampler_3d", linear_sampler_3d, 3)
_run_capacity_var_triplet("linear_affine_sampler_3d", linear_affine_sampler_3d, 3)
_run_capacity_var_triplet("sine_sampler", sine_sampler, 1)
_run_capacity_var_triplet("affine_sine_sampler", affine_sine_sampler, 1)
_run_capacity_var_triplet("cos_exp_sampler_2d", cos_exp_sampler_2d, 2)

NameError: name 'linear_sampler' is not defined